In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sqlalchemy import create_engine
import os

RAW       = Path(os.getcwd())   # same folder as your CSVs
PROCESSED = RAW / "processed"
DB_PATH   = RAW / "bluestock_mf.db"

PROCESSED.mkdir(exist_ok=True)
print("Processed folder:", PROCESSED)
print("DB will be saved at:", DB_PATH)

Processed folder: C:\Users\Hp\processed
DB will be saved at: C:\Users\Hp\bluestock_mf.db


In [2]:
df_nav = pd.read_csv(RAW / "02_nav_history.csv")

# Parse dates
df_nav["date"] = pd.to_datetime(df_nav["date"])

# Sort
df_nav = df_nav.sort_values(["amfi_code", "date"]).reset_index(drop=True)

# Remove duplicates
df_nav = df_nav.drop_duplicates(subset=["amfi_code", "date"])

# Remove invalid NAV
df_nav = df_nav[df_nav["nav"] > 0]

# Compute daily return %
df_nav["daily_return_pct"] = (
    df_nav.groupby("amfi_code")["nav"]
          .pct_change() * 100
)

print("Shape:", df_nav.shape)
print("Nulls:\n", df_nav.isnull().sum())
print(df_nav.head())

df_nav.to_csv(PROCESSED / "clean_nav.csv", index=False)
print("\nSaved clean_nav.csv")

Shape: (46000, 4)
Nulls:
 amfi_code            0
date                 0
nav                  0
daily_return_pct    40
dtype: int64
   amfi_code       date       nav  daily_return_pct
0     100016 2022-01-03  520.4608               NaN
1     100016 2022-01-04  515.0971         -1.030568
2     100016 2022-01-05  521.7239          1.286515
3     100016 2022-01-06  515.7880         -1.137747
4     100016 2022-01-07  515.1639         -0.120999

Saved clean_nav.csv


In [3]:
df_txn = pd.read_csv(RAW / "08_investor_transactions.csv")

# Parse dates
df_txn["transaction_date"] = pd.to_datetime(df_txn["transaction_date"])

# Standardise transaction type
df_txn["transaction_type"] = df_txn["transaction_type"].str.strip().str.title()
print("Transaction Types:\n", df_txn["transaction_type"].value_counts())

# Remove invalid amounts
df_txn = df_txn[df_txn["amount_inr"] > 0]

# Check KYC
print("\nKYC Status:\n", df_txn["kyc_status"].value_counts())

# Remove duplicates
df_txn = df_txn.drop_duplicates()

print(f"\nClean shape: {df_txn.shape}")
df_txn.to_csv(PROCESSED / "clean_transactions.csv", index=False)
print("Saved clean_transactions.csv")

Transaction Types:
 transaction_type
Sip           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

KYC Status:
 kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

Clean shape: (32778, 13)
Saved clean_transactions.csv


In [4]:
df_perf = pd.read_csv(RAW / "07_scheme_performance.csv")

numeric_cols = [
    "return_1yr_pct", "return_3yr_pct", "return_5yr_pct",
    "sharpe_ratio", "alpha", "beta", "max_drawdown_pct", "expense_ratio_pct"
]

print("=== Summary Stats ===")
print(df_perf[numeric_cols].describe())

print("\n=== Negative Sharpe Funds ===")
print(df_perf[df_perf["sharpe_ratio"] < 0][["scheme_name", "sharpe_ratio"]])

print("\n=== Expense Ratio Out of Range ===")
bad = df_perf[(df_perf["expense_ratio_pct"] < 0.1) | (df_perf["expense_ratio_pct"] > 2.5)]
print(bad[["scheme_name", "expense_ratio_pct"]])

df_perf.to_csv(PROCESSED / "clean_performance.csv", index=False)
print("\nSaved clean_performance.csv")

=== Summary Stats ===
       return_1yr_pct  return_3yr_pct  return_5yr_pct  sharpe_ratio  \
count       40.000000       40.000000       40.000000     40.000000   
mean        14.376000       14.089000       14.516750      1.361750   
std          4.883023        4.617253        4.454021      1.475805   
min          4.260000        5.140000        5.430000      0.800000   
25%         11.735000       12.035000       12.340000      0.865000   
50%         14.620000       14.205000       14.185000      0.925000   
75%         16.392500       15.882500       17.585000      0.985000   
max         24.930000       23.390000       23.800000      7.680000   

           alpha       beta  max_drawdown_pct  expense_ratio_pct  
count  40.000000  40.000000         40.000000          40.000000  
mean    1.253500   0.873250        -19.200250           1.237000  
std     0.447412   0.224846          8.819164           0.386584  
min     0.510000   0.220000        -33.500000           0.550000  
25%

In [5]:
files = {
    "01_fund_master.csv":          "clean_fund_master.csv",
    "03_aum_by_fund_house.csv":    "clean_aum.csv",
    "04_monthly_sip_inflows.csv":  "clean_sip.csv",
    "05_category_inflows.csv":     "clean_category_inflows.csv",
    "06_industry_folio_count.csv": "clean_folio.csv",
    "09_portfolio_holdings.csv":   "clean_holdings.csv",
    "10_benchmark_indices.csv":    "clean_benchmark.csv",
}

for raw_name, clean_name in files.items():
    df = pd.read_csv(RAW / raw_name)
    df.to_csv(PROCESSED / clean_name, index=False)
    print(f"Saved {clean_name}: {df.shape}")

Saved clean_fund_master.csv: (40, 15)
Saved clean_aum.csv: (90, 5)
Saved clean_sip.csv: (48, 6)
Saved clean_category_inflows.csv: (144, 3)
Saved clean_folio.csv: (21, 6)
Saved clean_holdings.csv: (322, 8)
Saved clean_benchmark.csv: (8050, 3)


In [6]:
engine = create_engine(f"sqlite:///{DB_PATH}")

tables = {
    "dim_fund":          pd.read_csv(PROCESSED / "clean_fund_master.csv"),
    "fact_nav":          pd.read_csv(PROCESSED / "clean_nav.csv"),
    "fact_transactions": pd.read_csv(PROCESSED / "clean_transactions.csv"),
    "fact_performance":  pd.read_csv(PROCESSED / "clean_performance.csv"),
    "fact_aum":          pd.read_csv(PROCESSED / "clean_aum.csv"),
    "fact_sip":          pd.read_csv(PROCESSED / "clean_sip.csv"),
    "fact_holdings":     pd.read_csv(PROCESSED / "clean_holdings.csv"),
    "fact_benchmark":    pd.read_csv(PROCESSED / "clean_benchmark.csv"),
}

for table_name, df in tables.items():
    df.to_sql(table_name, engine, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df)} rows")

print("\nDatabase ready at:", DB_PATH)

Loaded dim_fund: 40 rows
Loaded fact_nav: 46000 rows
Loaded fact_transactions: 32778 rows
Loaded fact_performance: 40 rows
Loaded fact_aum: 90 rows
Loaded fact_sip: 48 rows
Loaded fact_holdings: 322 rows
Loaded fact_benchmark: 8050 rows

Database ready at: C:\Users\Hp\bluestock_mf.db


In [7]:
import sqlite3

conn = sqlite3.connect(DB_PATH)

queries = {
    "1. Top 5 funds by AUM": """
        SELECT scheme_name, aum_crore 
        FROM fact_performance ORDER BY aum_crore DESC LIMIT 5""",

    "2. Avg NAV per month (SBI Bluechip)": """
        SELECT strftime('%Y-%m', date) as month, ROUND(AVG(nav),2) as avg_nav
        FROM fact_nav WHERE amfi_code = 119551
        GROUP BY month ORDER BY month LIMIT 10""",

    "3. SIP inflow by year": """
        SELECT substr(month,1,4) as year, ROUND(SUM(sip_inflow_crore),0) as total_crore
        FROM fact_sip GROUP BY year ORDER BY year""",

    "4. Transactions by state": """
        SELECT state, COUNT(*) as txn_count, ROUND(SUM(amount_inr)/10000000.0,2) as total_crore
        FROM fact_transactions GROUP BY state ORDER BY txn_count DESC""",

    "5. Funds with expense ratio < 1%": """
        SELECT scheme_name, fund_house, expense_ratio_pct 
        FROM dim_fund WHERE expense_ratio_pct < 1.0 ORDER BY expense_ratio_pct""",

    "6. Top 5 funds by Sharpe Ratio": """
        SELECT scheme_name, sharpe_ratio, return_3yr_pct 
        FROM fact_performance ORDER BY sharpe_ratio DESC LIMIT 5""",

    "7. Avg alpha by category": """
        SELECT category, ROUND(AVG(alpha),3) as avg_alpha 
        FROM fact_performance GROUP BY category ORDER BY avg_alpha DESC""",

    "8. Transactions by type": """
        SELECT transaction_type, COUNT(*) as count, 
               ROUND(SUM(amount_inr)/10000000.0,2) as total_crore
        FROM fact_transactions GROUP BY transaction_type""",

    "9. AUM by fund house latest": """
        SELECT fund_house, ROUND(MAX(aum_lakh_crore),2) as latest_aum_lakh_crore
        FROM fact_aum GROUP BY fund_house ORDER BY latest_aum_lakh_crore DESC""",

    "10. SIP inflow Dec 2025": """
        SELECT month, sip_inflow_crore, active_sip_accounts_crore 
        FROM fact_sip WHERE month = '2025-12'""",
}

for qname, sql in queries.items():
    print(f"\n{'='*50}")
    print(qname)
    result = pd.read_sql(sql, conn)
    print(result.to_string(index=False))

conn.close()


1. Top 5 funds by AUM
                                          scheme_name  aum_crore
Mirae Asset Emerging Bluechip Fund - Regular - Growth      49046
        Kotak Emerging Equity Fund - Regular - Growth      47469
       Nippon India Small Cap Fund - Regular - Growth      43630
           DSP Top 100 Equity Fund - Regular - Growth      41828
                  UTI Mid Cap Fund - Regular - Growth      41728

2. Avg NAV per month (SBI Bluechip)
  month  avg_nav
2022-01    55.09
2022-02    52.48
2022-03    50.84
2022-04    52.04
2022-05    51.89
2022-06    52.68
2022-07    52.69
2022-08    53.77
2022-09    56.55
2022-10    57.88

3. SIP inflow by year
year  total_crore
2022     149437.0
2023     184763.0
2024     269781.0
2025     335740.0

4. Transactions by state
         state  txn_count  total_crore
        Punjab       2965        31.58
Madhya Pradesh       2931        30.83
    Tamil Nadu       2806        31.52
       Gujarat       2780        29.84
   West Bengal       2748    